In [2]:
import polars as pl 
import polars_ds as pds 
import requests 
import json 
import duckdb
from pathlib import Path
from src.utils import scrape_ticket_sections, get_available_events, scrape_match_results, scrape_eliteserien_results,create_table_after_round

In [3]:
# List all table names
db_path = Path("data/brann.duckdb")
con = duckdb.connect(str(db_path))
table_names = con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'").fetchall()
print("Available tables:")
for table in table_names:
    print(f"  - {table[0]}")
con.close()

Available tables:
  - fct_league_standings
  - fct_matches
  - raw_eliteserien_results


In [10]:
con = duckdb.connect('data/brann.duckdb')
latest = pl.from_arrow(con.execute("""
    SELECT * FROM fct_matches 
""").arrow())
con.close()
latest

date,matchday,home_team,away_team,result,home_goals,away_goals,winner,snapshot_at,ingested_at
date,i64,str,str,str,i32,i32,str,"datetime[μs, Europe/Oslo]",datetime[μs]
2026-03-14,1,"""HamKam""","""Viking FK""","""2:1""",2,1,"""home_team""",2026-09-01 10:14:58.553514 CEST,2026-09-01 10:14:58.591575
2026-03-14,1,"""Molde FK""","""Rosenborg BK""","""2:0""",2,0,"""home_team""",2026-09-01 10:14:58.553514 CEST,2026-09-01 10:14:58.591575
2026-03-15,1,"""Kristiansund BK""","""SK Brann""","""3:2""",3,2,"""home_team""",2026-09-01 10:14:58.553514 CEST,2026-09-01 10:14:58.591575
2026-03-15,1,"""KFUM Oslo""","""IK Start""","""2:0""",2,0,"""home_team""",2026-09-01 10:14:58.553514 CEST,2026-09-01 10:14:58.591575
2026-03-15,1,"""Vålerenga""","""Sandefjord""","""1:0""",1,0,"""home_team""",2026-09-01 10:14:58.553514 CEST,2026-09-01 10:14:58.591575
…,…,…,…,…,…,…,…,…,…
2026-08-30,19,"""IK Start""","""KFUM Oslo""","""4:1""",4,1,"""home_team""",2026-09-01 10:14:58.553514 CEST,2026-09-01 10:14:58.591575
2026-08-30,19,"""Tromsø IL""","""Sarpsborg 08""","""0:0""",0,0,"""draw""",2026-09-01 10:14:58.553514 CEST,2026-09-01 10:14:58.591575
2026-08-30,19,"""Viking FK""","""Aalesunds FK""","""2:1""",2,1,"""home_team""",2026-09-01 10:14:58.553514 CEST,2026-09-01 10:14:58.591575


In [11]:
con = duckdb.connect('data/brann.duckdb')
latest = pl.from_arrow(con.execute("""
    SELECT * FROM fct_league_standings 
""").arrow())
con.close()

latest

matchday,team,total_points,total_goals_for,total_goals_against,goal_difference,position
i64,str,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]",i64
19,"""Bodø/Glimt""",44,47,14,33,1
19,"""Viking FK""",43,42,18,24,2
19,"""Tromsø IL""",35,34,20,14,3
19,"""Molde FK""",30,36,29,7,4
19,"""SK Brann""",26,36,27,9,5
…,…,…,…,…,…,…
1,"""SK Brann""",0,2,3,-1,12
1,"""Rosenborg BK""",0,0,2,-2,13
1,"""Aalesunds FK""",0,1,3,-2,14


In [9]:


# Connect to DuckDB and read raw_eliteserien_results
db_path = Path("data/brann.duckdb")
con = duckdb.connect(str(db_path))
eliteserien_db = pl.from_arrow(con.execute("SELECT * FROM raw_eliteserien_results").arrow())
con.close()

print(f"Loaded {len(eliteserien_db)} records from DuckDB")
eliteserien_db.head()

Loaded 144 records from DuckDB


C:\Users\Yafee Ishraq\AppData\Local\Temp\ipykernel_16268\854702497.py:4: FutureWarning: from_arrow(<ArrowStreamExportable>) will return a Series instead of a DataFrame in 2.0. To avoid this warning, pass the ArrowStreamExportable to either `pl.DataFrame` or `pl.Series` instead based on your desired output type.
  eliteserien_db = pl.from_arrow(con.execute("SELECT * FROM raw_eliteserien_results").arrow())


date,matchday,home_team,away_team,result,snapshot_at,ingested_at
date,i64,str,str,str,"datetime[μs, Europe/Oslo]",datetime[μs]
2026-03-14,1,"""HamKam""","""Viking FK""","""2:1""",2026-09-01 10:14:58.553514 CEST,2026-09-01 10:14:58.591575
2026-03-14,1,"""Molde FK""","""Rosenborg BK""","""2:0""",2026-09-01 10:14:58.553514 CEST,2026-09-01 10:14:58.591575
2026-03-15,1,"""Kristiansund BK""","""SK Brann""","""3:2""",2026-09-01 10:14:58.553514 CEST,2026-09-01 10:14:58.591575
2026-03-15,1,"""KFUM Oslo""","""IK Start""","""2:0""",2026-09-01 10:14:58.553514 CEST,2026-09-01 10:14:58.591575
2026-03-15,1,"""Vålerenga""","""Sandefjord""","""1:0""",2026-09-01 10:14:58.553514 CEST,2026-09-01 10:14:58.591575


In [2]:
available_events = get_available_events()

In [3]:
for event in available_events:
    print(event['event_id'])


1085523
1187151
1188514


In [18]:
eliteserien_results = scrape_eliteserien_results(season_id = 2025,year = 2026)

In [19]:
pl.DataFrame(eliteserien_results)

date,matchday,home_team,away_team,result,snapshot_at
date,i64,str,str,str,"datetime[μs, UTC]"
2026-03-14,1,"""HamKam""","""Viking FK""","""2:1""",2026-09-01 09:53:45.721672 UTC
2026-03-14,1,"""Molde FK""","""Rosenborg BK""","""2:0""",2026-09-01 09:53:45.721672 UTC
2026-03-15,1,"""Kristiansund BK""","""SK Brann""","""3:2""",2026-09-01 09:53:45.721672 UTC
2026-03-15,1,"""KFUM Oslo""","""IK Start""","""2:0""",2026-09-01 09:53:45.721672 UTC
2026-03-15,1,"""Vålerenga""","""Sandefjord""","""1:0""",2026-09-01 09:53:45.721672 UTC
…,…,…,…,…,…
2026-08-30,19,"""IK Start""","""KFUM Oslo""","""4:1""",2026-09-01 09:53:45.721672 UTC
2026-08-30,19,"""Tromsø IL""","""Sarpsborg 08""","""0:0""",2026-09-01 09:53:45.721672 UTC
2026-08-30,19,"""Viking FK""","""Aalesunds FK""","""2:1""",2026-09-01 09:53:45.721672 UTC


In [3]:
create_table_after_round(eliteserien_results)

matchday,team,points,goals_for,goals_against,goal_difference,total_points,total_goals_for,total_goals_against,total_goal_difference,table_position
i64,str,i32,i64,i64,i64,i32,i64,i64,i64,u32
1,"""HamKam""",3,2,1,1,3,2,1,1,1
1,"""KFUM Oslo""",3,2,0,2,3,2,0,2,1
1,"""Kristiansund BK""",3,3,2,1,3,3,2,1,1
1,"""Lillestrøm SK""",3,3,1,2,3,3,1,2,1
1,"""Molde FK""",3,2,0,2,3,2,0,2,1
…,…,…,…,…,…,…,…,…,…,…
18,"""KFUM Oslo""",1,1,1,0,19,19,27,-8,12
18,"""Sandefjord""",3,2,1,1,18,15,23,-8,13
18,"""Aalesunds FK""",1,5,5,0,15,27,41,-14,14


In [4]:
paok = scrape_ticket_sections(1187151)

In [22]:
from src.agent import run_question

run_question("Hvor mange kamper vant Brann i vårsesongen (bruk date kolonnen)?")

ValueError: Spørringen bruker en tabell som agenten ikke har tilgang til.

In [18]:
from src.agent import agent

for event in agent.stream(
    {"question": "Hvilke hjemmekamp tapte Brann med flest mål?"},
    stream_mode="updates",
):
    print(event)

{'generate_sql': {'sql': "SELECT \n    date,\n    matchday,\n    home_team,\n    away_team,\n    result,\n    home_goals - away_goals AS goal_difference\nFROM fct_matches\nWHERE home_team = 'SK Brann' AND winner = 'away_team'\nORDER BY goal_difference DESC\nLIMIT 1"}}
{'execute_sql': {'columns': ['date', 'matchday', 'home_team', 'away_team', 'result', 'goal_difference'], 'rows': [(datetime.date(2026, 3, 22), 2, 'SK Brann', 'Tromsø IL', '1:2', -1)]}}
